In [1]:
!pip install -q uv

!uv pip install --system \
    "vllm==0.6.5" \
    "transformers==4.45.2" \
    "accelerate==0.34.2" \
    "autoawq==0.2.9" \
    "httpx==0.27.2" \
    "openai==1.58.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 30.1 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 125 packages in 1.89s
Prepared 48 packages in 1m 05s
Uninstalled 25 packages in 4.48s
Installed 48 packages in 775ms
 - accelerate==1.13.0
 + accelerate==0.34.2
 + airportsdata==20260902
 + apache-tvm-ffi==0.1.13.post3
 + astor==0.8.1
 + autoawq==0.2.9
 + blake3==1.0.9
 + compressed-tensors==0.8.1
 + depyf==0.18.0
 + diskcache==5.6.3
 + gguf==0.10.0
 - httpx==0.28.1
 + httpx==0.27.2
 - huggingface-hub==1.8.0
 + huggingface-hub==0.36.2
 + interegular==0.3.3
 + lm-format-enforcer==0.10.12
 + mistral-common==1.11.7
 + msgspec==0.21.1
 - numpy==2.0.2
 + numpy==1.26.4
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu1

In [2]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
import sys, os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,        # bare flag
    "--tool-call-parser": "hermes",           # Qwen2.5 family uses hermes
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 933, logging to /content/server.log


In [4]:
with open("/content/server.log", "r") as f:
    print("".join(f.readlines()[-20:]))

In [5]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader
# note the number; compare to yesterday's fp16 resident VRAM

3 MiB


In [7]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [8]:
# Function-calling smoke test for Lab W3D4 (quantise and lock).
# Given in full. You run it; you do not write it. Paste the whole file as one
# Colab cell (after the tool-call-enabled vLLM server is healthy), then call
# run_smoke(base_url=..., model=...).
#
# What it does: fires 3 canonical prompts, k times each for n=10 total
# attempts - 8 that want a tool call, 2 distractors that must NOT call - and
# scores each attempt's BEHAVIOUR: a valid parseable tool_calls when one is
# wanted, or a clean refusal on the distractor.
# Gate: PASS if at least 8 of 10 attempts show correct behaviour AND the
# distractor stays call-free in the majority of its attempts. A model that
# always calls a tool fails the real consumer, so restraint is scored.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same test works against
# any team's service. No secrets: the local vLLM server needs no key.

from openai import OpenAI

# Two tools the model may call. Shapes match the OpenAI tools schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

# The 3 canonical prompts. Each carries how many attempts (k) it gets and how many
# tool calls a correct answer makes. n = sum of k = 10.
#   two_tool:   needs BOTH tools (weather + calculator)   -> expect >= 1 call
#   single:     needs ONE tool                            -> expect >= 1 call
#   distractor: needs NO tool, must NOT call one          -> expect 0 calls
CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    """Return the parsed tool_calls list on a response message, or []."""
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    """A tool call is valid if it names a known function and its arguments
    parse as JSON with the required field present."""
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    """Run the smoke test. Returns a result dict with counts and the pass gate."""
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0          # attempts that returned >=1 valid tool call
    distractor_attempts = 0
    distractor_call_free = 0         # distractor attempts that made NO tool call
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                # a "wants a call" prompt counts toward the 8/10 gate when it
                # returns at least one valid tool call
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                # the distractor counts toward the 8/10 gate when it correctly
                # makes NO tool call, and separately toward distractor compliance
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    # gate: >=8/10 correct behaviours AND distractor call-free in the majority
    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,       # 10
        "score": valid_call_attempts,           # correct behaviours, out of 10
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }


In [9]:
# paste the contents of smoke_test.py, then:
result = run_smoke(base_url="http://localhost:8000/v1",
                   model="Qwen/Qwen2.5-1.5B-Instruct-AWQ")
print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [10]:
import json
with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

In [12]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it. It reads
# ab_report.json and checks the schema, that vLLM's concurrency-8 throughput
# beats Monday's batch-8 baseline, and that the speedup fields were computed.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


baseline batch-8: 106.1, vllm concurrency-8: 270.3
speedup at 8: 2.55x
GREEN CHECK: PASS
